### RAG Pipeline -Data ingestion to vector DB

In [9]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pathlib as Path


In [10]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

def process_all_pdf(pdf_direc):
    all_documents = []

    pdf_dir = Path(pdf_direc)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files")

    for pdf_file in pdf_files:
        loader = PyPDFLoader(str(pdf_file))
        documents = loader.load()

        for doc in documents:
            doc.metadata.update({
                "source": str(pdf_file),
                "file_name": pdf_file.name,
                "document_id": pdf_file.stem,
                "page_number": doc.metadata.get("page", 0) + 1
            })

        all_documents.extend(documents)

    return all_documents


# Process all PDFs in the data directory
data_dir = "../Data"
if Path(data_dir).exists():
    documents = process_all_pdf(data_dir)

    print(f"Total pages loaded: {len(documents)}")
    if documents:
        print(documents[0].metadata)
    else:
        print("No documents were processed.")
else:
    print(f"Directory not found: {data_dir}")


Found 2 PDF files
Total pages loaded: 100
{'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.3 (Macintosh)', 'creationdate': '2026-06-10T14:28:17-06:00', 'moddate': '2026-06-10T14:28:27-06:00', 'trapped': '/False', 'source': '..\\Data\\pdf1.pdf', 'total_pages': 51, 'page': 0, 'page_label': '1', 'file_name': 'pdf1.pdf', 'document_id': 'pdf1', 'page_number': 1}


In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents):
    """
    Split documents into smaller chunks for RAG.
    """

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        separators=["\n\n", "\n", " ", ""]
    )

    chunks = text_splitter.split_documents(documents)

    print(f"Created {len(chunks)} chunks")

    return chunks
    print(chunks)